In [5]:
# Specify input and output files
pdb_file    = "/home/pkwon2/cage/1_diffusion/input/BCDE.pdb"
output_file = "/home/pkwon2/cage/1_diffusion/input/BCDE_multichain.pdb"

import os
from Bio.PDB import PDBParser, PDBIO, Superimposer

# Parse the structure
t_parser  = PDBParser(QUIET=True)
structure = t_parser.get_structure('orig', pdb_file)
model     = structure[0]

# Debug: list available chain IDs
available_chains = [chain.id for chain in model]
print("Available chains:", available_chains)

# Ensure required chains exist
for cid in ('B', 'E'):
    if cid not in model:
        raise KeyError(f"Chain '{cid}' not found. Available: {available_chains}")
chain_B = model['B']
chain_E = model['E']

# Extract CA atoms from matching residues
res_B = [r for r in chain_B if r.get_id()[0]==' ' and 'CA' in r]
res_E = [r for r in chain_E if r.get_id()[0]==' ' and 'CA' in r]
if len(res_B) != len(res_E):
    raise ValueError(f"Residue mismatch B({len(res_B)}) vs E({len(res_E)})")
fixed_atoms  = [r['CA'] for r in res_E]
mobile_atoms = [r['CA'] for r in res_B]

# Compute superposition
t_sup = Superimposer()
t_sup.set_atoms(fixed_atoms, mobile_atoms)
rot, trans = t_sup.rotran

# Inverse transform
inv_rot   = rot.T
inv_trans = -inv_rot.dot(trans)

print("Rotation matrix:\n", rot)
print("Translation vector:\n", trans)

# Define original and new chain ID mappings
original_ids   = ['B', 'C', 'D', 'E']
positive_ids   = ['F', 'G', 'H', 'I']
negative_ids   = ['J', 'K', 'L', 'M']
transforms = [
    (rot, trans, positive_ids),
    (inv_rot, inv_trans, negative_ids)
]

# Apply transforms to each original chain and append
iow = PDBIO()
for (r_mat, t_vec, new_ids) in transforms:
    # Reload fresh structure for source chains
    temp_struct = t_parser.get_structure('tmp', pdb_file)
    temp_model  = temp_struct[0]
    for orig_id, new_id in zip(original_ids, new_ids):
        if orig_id not in temp_model:
            raise KeyError(f"Chain '{orig_id}' missing in source file")
        chain = temp_model[orig_id]
        for atom in chain.get_atoms():
            atom.set_coord(r_mat.dot(atom.get_coord()) + t_vec)
        chain.id = new_id
        model.add(chain)

# Save out the expanded multichain PDB
iow.set_structure(structure)
iow.save(output_file)
print(f"Written multi-chain PDB with FGHI and JKLM to: {output_file}")

Available chains: ['B', 'C', 'D', 'E']
Rotation matrix:
 [[-9.99999979e-01  1.75278978e-06 -2.07068410e-04]
 [-1.75548822e-06 -1.00000000e+00  1.30314452e-05]
 [-2.07068388e-04  1.30318085e-05  9.99999978e-01]]
Translation vector:
 [17.36136633 -3.98483168 25.84267865]
Written multi-chain PDB with FGHI and JKLM to: /home/pkwon2/cage/1_diffusion/input/BCDE_multichain.pdb


In [7]:
# Specify parameters in a notebook cell
input_dir   = "/home/pkwon2/cage/1_diffusion/outputs"
output_dir  = "/home/pkwon2/cage/1_diffusion/for_homo_mpnn"

import os
import glob
from Bio.PDB import PDBParser, PDBIO, Superimposer

# Ensure output directory exists
os.makedirs(output_dir, exist_ok=True)

parser = PDBParser(QUIET=True)
io     = PDBIO()

for pdb_path in glob.glob(os.path.join(input_dir, '*.pdb')):
    name      = os.path.basename(pdb_path)
    structure = parser.get_structure('orig', pdb_path)
    model     = structure[0]

    # --- 1. Grab chains A & B and compute transforms ---
    chain_A = model['A']
    chain_B = model['B']

    # only standard residues
    resA = [r for r in chain_A if r.get_id()[0] == ' ']
    resB = [r for r in chain_B if r.get_id()[0] == ' ']

    if len(resA) < 92 or len(resB) < 92:
        print(f"Skipping {name}: need ≥92 standard residues in both A & B")
        continue

    fixed_atoms  = [r['CA'] for r in resA[:92]]    # first 92 of A
    mobile_atoms = [r['CA'] for r in resB[-92:]]   # last 92 of B

    sup = Superimposer()
    sup.set_atoms(fixed_atoms, mobile_atoms)
    rot, trans = sup.rotran

    # inverse transform
    inv_rot   = rot.T
    inv_trans = -inv_rot.dot(trans)

    # --- 2. Detach only chain A, keep B in place ---
    model.detach_child('A')

    # --- 3. Build two new chains from original B ---
    for R, T, new_id in [
        (rot,       trans,    'D'),  # forward → D
        (inv_rot,   inv_trans,'E')   # inverse → E
    ]:
        # reload original B so we don't disturb the kept B
        tmp_struct = parser.get_structure(f"tmp{new_id}", pdb_path)
        tmp_chain  = tmp_struct[0]['B']

        # apply transform
        for atom in tmp_chain.get_atoms():
            atom.set_coord(R.dot(atom.get_coord()) + T)

        # rename and append
        tmp_chain.id = new_id
        model.add(tmp_chain)

    # --- 4. Save chain B + D + E ---
    out_file = os.path.join(output_dir, name)
    io.set_structure(structure)
    io.save(out_file)
    print(f"Wrote chains B, D & E to {out_file}")

Wrote chains B, D & E to /home/pkwon2/cage/1_diffusion/for_homo_mpnn/fiber_to_lhda_value1_105_value2_25_value3_80_ckpt_BFF_0_0_refined_1.pdb
Wrote chains B, D & E to /home/pkwon2/cage/1_diffusion/for_homo_mpnn/fiber_to_lhda_value1_107_value2_25_value3_80_ckpt_BFF_0_0_refined_2.pdb
Wrote chains B, D & E to /home/pkwon2/cage/1_diffusion/for_homo_mpnn/fiber_to_lhda_value1_108_value2_25_value3_80_ckpt_BFF_0_0_refined_1.pdb
Wrote chains B, D & E to /home/pkwon2/cage/1_diffusion/for_homo_mpnn/fiber_to_lhda_value1_106_value2_25_value3_80_ckpt_BFF_0_0_refined_0.pdb
Wrote chains B, D & E to /home/pkwon2/cage/1_diffusion/for_homo_mpnn/fiber_to_lhda_value1_102_value2_25_value3_80_ckpt_BFF_0_0_refined_1.pdb
Wrote chains B, D & E to /home/pkwon2/cage/1_diffusion/for_homo_mpnn/fiber_to_lhda_value1_101_value2_25_value3_80_ckpt_BFF_0_0_refined_0.pdb
Wrote chains B, D & E to /home/pkwon2/cage/1_diffusion/for_homo_mpnn/fiber_to_lhda_value1_100_value2_25_value3_80_ckpt_BFF_0_0_refined_2.pdb
Wrote chains 